In [1]:
import os, json, math, time, random, zipfile, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.checkpoint import checkpoint
from huggingface_hub import hf_hub_download, HfApi
from sklearn.metrics import f1_score, classification_report
from kaggle_secrets import UserSecretsClient

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

hf_token = UserSecretsClient().get_secret("HF")
data_repo = "Exploration-Lab/CS-781-Capstone"
model_repo = "Granth2/cs781-phase1-Submission-9"
tokenizer_repo = "Granth2/cs781-custom-bpe-30k-Version-10-40-40-10"
output_dir = "/kaggle/working/phase3"
os.makedirs(output_dir, exist_ok=True)

device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

lorma_variant = "plus"
lorma_rank = 8
lorma_alpha = 16.0
target_layers = ("attn.q_proj", "attn.v_proj")

use_bigru = True
gru_hidden = 256
dropout_rate = 0.2
grad_checkpoint = False
max_tokens = 128

window_size = 32
batch_windows = 2
accum_steps = 2
eval_batch_size = 64

num_epochs = 10
patience = 10
lr_lorma = 3e-4
lr_head = 1e-3
weight_decay = 0.01
warmup_frac = 0.06
class_weight_power = 0.5
max_grad_norm = 1.0
log_every = 25

use_kd = True
num_student_layers = 12
kd_temperature = 2.0
kd_alpha = 0.5
feature_weight = 0.5
retrain_teacher = True
teacher_save_path = os.path.join(output_dir, "teacher_trainable.pt")

In [2]:
%%writefile attention.py
import math
from typing import Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.1, attention_scale: float = None,
                 use_rope: bool = True, rope_base: float = 10000.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be evenly divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.gate_proj = nn.Linear(d_model, d_model, bias=False)

        self.use_rope = use_rope
        if use_rope:
            assert self.d_head % 2 == 0, "RoPE needs an even head dimension"
            inv_freq = 1.0 / (rope_base ** (torch.arange(0, self.d_head, 2, dtype=torch.float32) / self.d_head))
            angles = torch.arange(max_seq_len, dtype=torch.float32).unsqueeze(1) * inv_freq.unsqueeze(0)
            self.register_buffer("rope_cos", torch.cos(angles), persistent=False)
            self.register_buffer("rope_sin", torch.sin(angles), persistent=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)),
            persistent=False
        )

    def _apply_rope(self, x: torch.Tensor, start: int) -> torch.Tensor:
        T = x.size(2)
        cos = self.rope_cos[start:start + T].to(torch.float32)
        sin = self.rope_sin[start:start + T].to(torch.float32)
        x1, x2 = x[..., : self.d_head // 2].float(), x[..., self.d_head // 2:].float()
        out = torch.cat((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1)
        return out.to(x.dtype)

    def forward(
        self,
        x: torch.Tensor,
        past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
        key_padding_mask: Optional[torch.Tensor] = None,
    ):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        past_len = 0
        if past_kv is not None:
            past_len = past_kv[0].size(2)

        if self.use_rope:
            q = self._apply_rope(q, past_len)
            k = self._apply_rope(k, past_len)

        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        total_len = past_len + T
        present = (k, v) if use_cache else None

        if key_padding_mask is None and past_len == 0:
            attn_output = F.scaled_dot_product_attention(
                q, k, v,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=True,
                scale=self.attention_scale,
            )
        else:
            attn_scores = (q @ k.transpose(-2, -1)) * self.attention_scale

            mask = self.causal_mask[past_len:total_len, :total_len]
            if key_padding_mask is not None:
                mask = mask.unsqueeze(0) & key_padding_mask.to(torch.bool).unsqueeze(1)
                mask = mask | ~mask.any(dim=-1, keepdim=True)
                mask = mask.unsqueeze(1)
            attn_scores = attn_scores.masked_fill(~mask, float("-inf"))

            attn_probs = F.softmax(attn_scores, dim=-1)
            attn_probs = self.attn_dropout(attn_probs)

            attn_output = attn_probs @ v


        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, C)
        attn_output = attn_output * torch.sigmoid(self.gate_proj(x))
        attn_output = self.out_proj(attn_output)
        attn_output = self.resid_dropout(attn_output)

        if use_cache:
            return attn_output, present
        return attn_output

Writing attention.py


In [3]:
%%writefile model_arch.py
import math
from dataclasses import dataclass
from typing import Optional, List, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.utils import ModelOutput
from attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 560
    n_layers: int = 8
    n_heads: int = 8
    d_ff: int = 2240
    max_seq_len: int = 1024
    dropout: float = 0
    pos_emb_type: str = "rope"
    attention_scale: float = None
    norm_first: bool = True
    bos_token_id: int = None
    eos_token_id: int = None


@dataclass
class CausalLMOutput(ModelOutput):
    loss: Optional[torch.Tensor] = None
    logits: Optional[torch.Tensor] = None
    past_key_values: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model, dtype=torch.float32)
    pe[:, 0::2] = torch.sin(pos * div_term)
    pe[:, 1::2] = torch.cos(pos * div_term)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.norm_first = getattr(cfg, 'norm_first', True)
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(
            d_model=cfg.d_model,
            n_heads=cfg.n_heads,
            max_seq_len=cfg.max_seq_len,
            dropout=cfg.dropout,
            attention_scale=cfg.attention_scale,
            use_rope=(cfg.pos_emb_type == "rope")
        )
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor, past_kv=None, use_cache: bool = False, key_padding_mask=None):
        if self.norm_first:
            attn_out = self.attn(self.ln1(x), past_kv=past_kv, use_cache=use_cache, key_padding_mask=key_padding_mask)
            if use_cache:
                attn_out, present = attn_out
            x = x + attn_out
            x = x + self.ffn(self.ln2(x))
        else:
            attn_out = self.attn(x, past_kv=past_kv, use_cache=use_cache, key_padding_mask=key_padding_mask)
            if use_cache:
                attn_out, present = attn_out
            x = self.ln1(x + attn_out)
            x = self.ln2(x + self.ffn(x))
        if use_cache:
            return x, present
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        self.head.weight = self.tok_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.ones_(module.weight)
            torch.nn.init.zeros_(module.bias)

    def forward(self, input_ids: torch.Tensor = None, labels: torch.Tensor = None, idx: torch.Tensor = None,
                past_key_values: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None,
                use_cache: bool = False,
                attention_mask: Optional[torch.Tensor] = None,
                last_token_only: bool = False):
        if idx is None:
            idx = input_ids
        assert idx is not None, "Either input_ids or idx must be provided"

        B, T = idx.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.cfg.max_seq_len, \
            f"Seq length {past_len + T} exceeds max sequence length {self.cfg.max_seq_len}"

        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            if attention_mask is None:
                x = x + self.pos_emb[past_len:past_len + T]
            else:
                pos_ids = (attention_mask.long().cumsum(dim=-1) - 1).clamp(min=0)[:, -T:]
                x = x + self.pos_emb[pos_ids]

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past_kv = None if past_key_values is None else past_key_values[i]
            if use_cache:
                x, present = block(x, past_kv=past_kv, use_cache=True, key_padding_mask=attention_mask)
                presents.append(present)
            else:
                x = block(x, past_kv=past_kv, key_padding_mask=attention_mask)

        if last_token_only:
            x = x[:, -1:, :]
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous().view(-1, self.cfg.vocab_size)
            shift_labels = labels[:, 1:].contiguous().view(-1)
            loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=-100)
            return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents)

        if use_cache:
            return logits, presents
        return logits

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0, top_p: float = 1.0, eos_token_id: int = None, attention_mask: torch.Tensor = None,
                 stop_check=None, check_every: int = 16) -> torch.Tensor:
        was_training = self.training
        self.eval()

        B = idx.size(0)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device=idx.device, dtype=torch.bool)
        finished = torch.zeros(B, dtype=torch.bool, device=idx.device)

        prompt_len = idx.size(1)
        past_key_values = None
        cond_mask = None
        for step in range(max_new_tokens):
            if past_key_values is None or past_key_values[0][0].size(2) + 1 > self.cfg.max_seq_len:
                past_key_values = None
                idx_cond = idx[:, -self.cfg.max_seq_len:]
                cond_mask = None if attention_mask is None else attention_mask[:, -self.cfg.max_seq_len:]
            else:
                idx_cond = idx[:, -1:]

            logits, past_key_values = self(idx_cond, past_key_values=past_key_values, use_cache=True,
                                           attention_mask=cond_mask, last_token_only=True)
            logits = logits[:, -1, :] / max(temperature, 1e-5)

            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
                logits = logits.masked_fill(indices_to_remove, float('-inf'))

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

            if eos_token_id is not None:
                idx_next = torch.where(finished.unsqueeze(1), torch.full_like(idx_next, eos_token_id), idx_next)
                finished = finished | (idx_next.squeeze(1) == eos_token_id)

            idx = torch.cat((idx, idx_next), dim=1)
            if attention_mask is not None:
                ones = torch.ones(B, 1, dtype=torch.bool, device=idx.device)
                attention_mask = torch.cat((attention_mask, ones), dim=1)
                cond_mask = torch.cat((cond_mask, ones), dim=1)

            if eos_token_id is not None and finished.all():
                break

            if stop_check is not None and (step + 1) % check_every == 0:
                satisfied = torch.as_tensor(stop_check(idx[:, prompt_len:]), dtype=torch.bool, device=idx.device)
                if (satisfied | finished).all():
                    break

        if was_training:
            self.train()
        return idx


    @staticmethod
    def _repeated_ngram_tokens(seq, n):
        if len(seq) < n - 1:
            return []
        prefix = tuple(seq[len(seq) - (n - 1):])
        banned = []
        for i in range(len(seq) - n + 1):
            if tuple(seq[i:i + n - 1]) == prefix:
                banned.append(seq[i + n - 1])
        return banned

    @torch.no_grad()
    def generate_beam(self, idx: torch.Tensor, max_new_tokens: int, num_beams: int = 4, temperature: float = 1.0,
                      attention_mask: torch.Tensor = None, no_repeat_ngram_size: int = 0, banned_token_ids=None,
                      stop_check=None, check_every: int = 16) -> torch.Tensor:
        was_training = self.training
        self.eval()

        B, T = idx.shape
        nb = num_beams
        device = idx.device
        max_new_tokens = min(max_new_tokens, self.cfg.max_seq_len - T)

        if attention_mask is None:
            attention_mask = torch.ones(B, T, dtype=torch.bool, device=device)
        attention_mask = attention_mask.to(device=device, dtype=torch.bool)

        logits, past = self(idx, use_cache=True, attention_mask=attention_mask, last_token_only=True)
        past = [(k.repeat_interleave(nb, dim=0), v.repeat_interleave(nb, dim=0)) for k, v in past]
        mask = attention_mask.repeat_interleave(nb, dim=0)
        next_logits = logits[:, -1, :].repeat_interleave(nb, dim=0)

        beam_scores = torch.zeros(B, nb, device=device)
        beam_scores[:, 1:] = float("-inf")
        gen = torch.empty(B * nb, 0, dtype=torch.long, device=device)
        row_offset = (torch.arange(B, device=device) * nb).unsqueeze(1)

        for step in range(max_new_tokens):
            if step > 0:
                mask = torch.cat([mask, torch.ones(B * nb, 1, dtype=torch.bool, device=device)], dim=1)
                out, past = self(gen[:, -1:], past_key_values=past, use_cache=True,
                                 attention_mask=mask, last_token_only=True)
                next_logits = out[:, -1, :]

            log_probs = F.log_softmax(next_logits.float() / max(temperature, 1e-5), dim=-1)

            if banned_token_ids:
                log_probs[:, banned_token_ids] = float("-inf")

            if no_repeat_ngram_size > 0 and gen.size(1) >= no_repeat_ngram_size - 1:
                for r, seq in enumerate(gen.tolist()):
                    bad = self._repeated_ngram_tokens(seq, no_repeat_ngram_size)
                    if bad:
                        log_probs[r, bad] = float("-inf")

            vocab = log_probs.size(-1)
            scores = (beam_scores.view(-1, 1) + log_probs).reshape(B, nb * vocab)
            beam_scores, top_idx = scores.topk(nb, dim=-1)

            src_beam = (top_idx // vocab + row_offset).view(-1)
            next_tok = (top_idx % vocab).view(-1, 1)

            gen = torch.cat([gen[src_beam], next_tok], dim=1)
            past = [(k[src_beam], v[src_beam]) for k, v in past]

            if stop_check is not None and (step + 1) % check_every == 0:
                best_so_far = gen.view(B, nb, -1)[:, 0]
                satisfied = torch.as_tensor(stop_check(best_so_far), dtype=torch.bool)
                if satisfied.all():
                    break

        best = gen.view(B, nb, -1)[:, 0]
        if was_training:
            self.train()
        return torch.cat([idx, best], dim=1)

Writing model_arch.py


In [4]:
import os
import json
from transformers import PreTrainedTokenizer


class MyCustomTokenizer(PreTrainedTokenizer):
    END_OF_WORD = "</w>"

    def __init__(self, vocab_file, merges_file, pad_token="<pad>", unk_token="<unk>",
                 bos_token="<s>", eos_token="</s>", **kwargs):
        with open(vocab_file, "r", encoding="utf-8") as f:
            self.vocab = json.load(f)
        self.id_to_token = {i: t for t, i in self.vocab.items()}

        self.byte_to_char = self._build_byte_to_char()
        self.char_to_byte = {c: b for b, c in self.byte_to_char.items()}

        self.merge_ranks = self._load_merges(merges_file)
        self.word_cache = {}

        kwargs.setdefault("clean_up_tokenization_spaces", False)
        super().__init__(
            pad_token=pad_token,
            unk_token=unk_token,
            bos_token=bos_token,
            eos_token=eos_token,
            **kwargs
        )

    @staticmethod
    def _build_byte_to_char():
        printable = (list(range(ord("!"), ord("~") + 1))
                     + list(range(ord("¡"), ord("¬") + 1))
                     + list(range(ord("®"), ord("ÿ") + 1)))
        byte_values = printable[:]
        char_codes = printable[:]
        extra = 0
        for b in range(256):
            if b not in printable:
                byte_values.append(b)
                char_codes.append(256 + extra)
                extra += 1
        return {b: chr(c) for b, c in zip(byte_values, char_codes)}

    def _byte_ids_to_symbol(self, text):
        symbol = ""
        for part in text.split("+"):
            if part == self.END_OF_WORD:
                symbol += part
            else:
                symbol += self.byte_to_char[int(part[2:])]
        return symbol

    def _load_merges(self, merges_file):
        pairs = []
        with open(merges_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#version"):
                    continue
                parts = line.split(" ")
                if len(parts) == 2:
                    pairs.append((parts[0], parts[1]))

        written_as_byte_ids = len(pairs) > 0 and pairs[0][0].startswith("b:")

        merge_ranks = {}
        for left, right in pairs:
            if written_as_byte_ids:
                left = self._byte_ids_to_symbol(left)
                right = self._byte_ids_to_symbol(right)
            merge_ranks[(left, right)] = len(merge_ranks)
        return merge_ranks

    @property
    def vocab_size(self):
        return len(self.vocab)

    def get_vocab(self):
        return self.vocab

    def _split_word(self, word):
        if word in self.word_cache:
            return self.word_cache[word]

        symbols = [self.byte_to_char[b] for b in word.encode("utf-8")] + [self.END_OF_WORD]

        while len(symbols) > 1:
            pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            known_pairs = [p for p in pairs if p in self.merge_ranks]
            if not known_pairs:
                break

            best_pair = min(known_pairs, key=lambda p: self.merge_ranks[p])

            new_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best_pair:
                    new_symbols.append(best_pair[0] + best_pair[1])
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols

        tokens = [s if s in self.vocab else self.unk_token for s in symbols]

        if len(self.word_cache) > 500_000:
            self.word_cache.clear()
        self.word_cache[word] = tokens
        return tokens

    def _tokenize(self, text):
        tokens = []
        for word in text.split():
            tokens.extend(self._split_word(word))
        return tokens

    def convert_tokens_to_string(self, tokens):
        byte_list = []
        for token in tokens:
            if token in self.all_special_tokens:
                continue
            word_ends = token.endswith(self.END_OF_WORD)
            if word_ends:
                token = token[:-len(self.END_OF_WORD)]
            byte_list.extend(self.char_to_byte[c] for c in token if c in self.char_to_byte)
            if word_ends:
                byte_list.append(ord(" "))
        return bytes(byte_list).decode("utf-8", errors="replace").strip()

    def _convert_token_to_id(self, token):
        return self.vocab.get(token, self.vocab.get(self.unk_token, 1))

    def _convert_id_to_token(self, index):
        return self.id_to_token.get(index, self.unk_token)

    def save_vocabulary(self, save_directory, filename_prefix=None):
        prefix = (filename_prefix + "-") if filename_prefix else ""
        vocab_file = os.path.join(save_directory, prefix + "vocab.json")
        merges_file = os.path.join(save_directory, prefix + "merges.txt")
        with open(vocab_file, "w", encoding="utf-8") as f:
            json.dump(self.vocab, f, ensure_ascii=False, indent=2)
        with open(merges_file, "w", encoding="utf-8") as f:
            f.write("#version: 0.2\n")
            for (left, right), _ in sorted(self.merge_ranks.items(), key=lambda item: item[1]):
                f.write(f"{left} {right}\n")
        return (vocab_file, merges_file)

In [5]:
class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha, variant):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        self.d = base.out_features
        self.r = r
        self.variant = variant
        self.scale = alpha / r

        self.A = nn.Parameter(torch.zeros(r, self.d))
        self.B = nn.Parameter(torch.zeros(self.d, r))

        with torch.no_grad():
            if variant == "plus":
                nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            else:
                self.A[0, 0] = 1.0
                self.B[:, 0] = 1.0
                nn.init.kaiming_uniform_(self.B[:, 1:], a=math.sqrt(5))
                shift = (torch.arange(self.d)[None, :] - torch.arange(self.d)[:, None]) % self.d
                self.register_buffer("shift", shift, persistent=False)

    def update_matrix(self):
        if self.variant == "plus":
            return self.scale * (self.B @ self.A) + torch.eye(self.d, device=self.A.device)
        return (self.B @ self.A).gather(1, self.shift)

    def forward(self, x):
        out = F.linear(x, self.base.weight)
        if self.variant == "plus":
            out = out + self.scale * F.linear(F.linear(out, self.A), self.B)
        else:
            out = F.linear(out, self.update_matrix())
        if self.base.bias is not None:
            out = out + self.base.bias
        return out

    @torch.no_grad()
    def merged_weight(self):
        return self.update_matrix() @ self.base.weight


def add_lorma(model, target_layers, r, alpha, variant):
    model.requires_grad_(False)
    to_wrap = []
    for module_name, module in model.named_modules():
        for child_name, child in module.named_children():
            full_name = f"{module_name}.{child_name}"
            if isinstance(child, nn.Linear) and full_name.endswith(target_layers):
                to_wrap.append((module, child_name))
    for module, child_name in to_wrap:
        setattr(module, child_name, LoRMALinear(getattr(module, child_name), r, alpha, variant))
    return len(to_wrap)


def merge_lorma(module):
    for name, child in module.named_children():
        if isinstance(child, LoRMALinear):
            merged = nn.Linear(child.base.in_features, child.base.out_features, bias=child.base.bias is not None)
            merged.weight.data = child.merged_weight().clone()
            if child.base.bias is not None:
                merged.bias.data = child.base.bias.data.clone()
            setattr(module, name, merged)
        else:
            merge_lorma(child)

In [6]:
def pad_sentences(sentences, pad_id):
    lengths = torch.tensor([len(s) for s in sentences])
    ids = torch.full((len(sentences), int(lengths.max())), pad_id, dtype=torch.long)
    for i, s in enumerate(sentences):
        ids[i, :len(s)] = torch.tensor(s)
    return ids, lengths


class RoleClassifier(nn.Module):
    def __init__(self, decoder, num_labels, use_bigru, gru_hidden, dropout, grad_checkpoint):
        super().__init__()
        self.decoder = decoder
        self.grad_checkpoint = grad_checkpoint
        dim = 2 * decoder.cfg.d_model
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        if use_bigru:
            self.gru = nn.GRU(dim, gru_hidden, batch_first=True, bidirectional=True)
            self.classifier = nn.Linear(2 * gru_hidden, num_labels)
        else:
            self.gru = None
            self.classifier = nn.Linear(dim, num_labels)

    def embed_sentences(self, ids, lengths):
        x = self.decoder.tok_emb(ids)
        for block in self.decoder.blocks:
            if self.grad_checkpoint and self.training:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        x = self.decoder.ln_f(x)

        mask = (torch.arange(x.size(1), device=x.device)[None, :] < lengths[:, None]).unsqueeze(-1)
        mean = (x * mask).sum(1) / lengths.unsqueeze(-1)
        last = x[torch.arange(x.size(0), device=x.device), lengths - 1]
        return torch.cat([mean, last], dim=-1)

    def run_gru(self, vectors, doc_lengths):
        x = self.dropout(self.norm(vectors))
        if self.gru is not None:
            packed = pack_padded_sequence(x, doc_lengths.cpu(), batch_first=True, enforce_sorted=False)
            x, _ = self.gru(packed)
            x, _ = pad_packed_sequence(x, batch_first=True, total_length=vectors.size(1))
        return self.classifier(self.dropout(x))

    def forward(self, ids, lengths, window_sizes, return_vectors=False):
        vectors = self.embed_sentences(ids, lengths)
        padded = pad_sequence(list(torch.split(vectors, window_sizes)), batch_first=True)
        logits = self.run_gru(padded, torch.tensor(window_sizes))
        if return_vectors:
            return logits, vectors
        return logits

    @torch.no_grad()
    def predict(self, sentences, pad_id, batch_size, window=None):
        self.eval()
        vectors = []
        for i in range(0, len(sentences), batch_size):
            ids, lengths = pad_sentences(sentences[i:i + batch_size], pad_id)
            vectors.append(self.embed_sentences(ids.to(device), lengths.to(device)).float())
        vectors = torch.cat(vectors)
        chunks = [vectors] if window is None else torch.split(vectors, window)
        return torch.cat([self.run_gru(c.unsqueeze(0), torch.tensor([len(c)]))[0] for c in chunks])

In [7]:
phase3_files = [f for f in HfApi().list_repo_files(data_repo, repo_type="dataset", token=hf_token)
                if f.startswith("Phase_3")]

def download(filename):
    path = [f for f in phase3_files if f.endswith("/" + filename)][0]
    return hf_hub_download(data_repo, path, repo_type="dataset", token=hf_token)

def load_docs(name):
    raw = json.load(open(download(name + ".json"), encoding="utf-8"))
    docs = []
    if isinstance(raw, dict):
        for doc_id, value in raw.items():
            if isinstance(value, dict):
                docs.append({"id": str(doc_id), "sents": value["text"], "labels": value.get("labels")})
            else:
                docs.append({"id": str(doc_id), "sents": value, "labels": None})
    else:
        for doc in raw:
            docs.append({"id": str(doc["id"]), "sents": doc["text"], "labels": doc.get("labels")})
    return docs


all_docs = {name: load_docs(name) for name in ["CL_train", "CL_dev", "CL_test", "IT_train", "IT_dev", "IT_test"]}

label_to_id = json.load(open(download("label_vocab.json")))
label_names = sorted(label_to_id, key=label_to_id.get)
num_labels = len(label_names)

for docs in all_docs.values():
    for doc in docs:
        if doc["labels"] is not None:
            doc["y"] = [label_to_id[label] for label in doc["labels"]]

train_docs = all_docs["CL_train"] + all_docs["IT_train"]
dev_docs = all_docs["CL_dev"] + all_docs["IT_dev"]
test_docs = all_docs["CL_test"] + all_docs["IT_test"]

for name, docs in all_docs.items():
    print(f"{name:<9s} {len(docs):>5d} docs  {sum(len(d['sents']) for d in docs):>7d} sentences")

label_counts = np.bincount([y for d in train_docs for y in d["y"]], minlength=num_labels)
for i in np.argsort(-label_counts):
    print(f"{label_names[i]:<24s}{label_counts[i]:>8d}  {100 * label_counts[i] / label_counts.sum():5.2f}%")

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_train     40 docs    10593 sentences
CL_dev        5 docs     1396 sentences
CL_test       5 docs     1339 sentences
IT_train     40 docs     6252 sentences
IT_dev        5 docs      746 sentences
IT_test       5 docs      858 sentences
RatioOfTheDecision          4301  25.53%
Fact                        4098  24.33%
PrecedentReliedUpon         1945  11.55%
ArgumentPetitioner          1613   9.58%
ArgumentRespondent          1348   8.00%
Statute                     1099   6.52%
RulingByLowerCourt           874   5.19%
PrecedentNotReliedUpon       569   3.38%
RulingByPresentCourt         307   1.82%
None                         297   1.76%
Dissent                      284   1.69%
Issue                        107   0.64%
PrecedentOverruled             3   0.02%


In [8]:
tokenizer = MyCustomTokenizer(
    vocab_file=hf_hub_download(tokenizer_repo, "vocab.json", token=hf_token),
    merges_file=hf_hub_download(tokenizer_repo, "merges.txt", token=hf_token),
)
pad_id = tokenizer.pad_token_id

for docs in all_docs.values():
    for doc in docs:
        tokens = [tokenizer.encode(s, add_special_tokens=False)[:max_tokens] for s in doc["sents"]]
        doc["ids"] = [t or [tokenizer.unk_token_id] for t in tokens]

lengths = np.array([len(t) for d in train_docs for t in d["ids"]])
print(f"tokens per sentence: median {int(np.median(lengths))}, 95th percentile {int(np.percentile(lengths, 95))}")
print(f"sentences cut at {max_tokens} tokens: {100 * (lengths == max_tokens).mean():.1f}%")

vocab.json:   0%|          | 0.00/640k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/320k [00:00<?, ?B/s]

tokens per sentence: median 31, 95th percentile 87
sentences cut at 128 tokens: 1.7%


In [9]:
from model_arch import DecoderLM, LMConfig

base_model_weights = torch.load(hf_hub_download(model_repo, "pytorch_model.bin", token=hf_token),
                          map_location="cpu")


def build_classifier(keep_layers):
    config = LMConfig(vocab_size=len(tokenizer), d_model=768, n_layers=12, n_heads=12,
                      d_ff=3072, max_seq_len=1024, dropout=0.0)
    decoder = DecoderLM(config)
    decoder.load_state_dict(base_model_weights)

    decoder.blocks = nn.ModuleList([decoder.blocks[i] for i in keep_layers])
    decoder.cfg.n_layers = len(keep_layers)

    classifier = RoleClassifier(decoder, num_labels, use_bigru, gru_hidden, dropout_rate, grad_checkpoint)
    n_wrapped = add_lorma(classifier.decoder, target_layers, lorma_rank, lorma_alpha, lorma_variant)
    classifier.to(device)

    n_total = sum(p.numel() for p in classifier.parameters())
    n_trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
    print(f"{len(keep_layers)} decoder layers {keep_layers} | LoRMA on {n_wrapped} linears | "
          f"trainable {n_trainable:,} of {n_total:,} ({100 * n_trainable / n_total:.2f}%)")
    return classifier


num_teacher_layers = 12
teacher = build_classifier(list(range(num_teacher_layers)))

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  461MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

12 decoder layers [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] | LoRMA on 24 linears | trainable 3,060,237 of 118,197,261 (2.59%)


In [10]:
train_windows = [(d, start, min(start + window_size, len(doc["ids"])))
           for d, doc in enumerate(train_docs) for start in range(0, len(doc["ids"]), window_size)]

def make_batch(batch_windows):
    sentences, window_sizes, labels = [], [], []
    for d, start, end in batch_windows:
        sentences += train_docs[d]["ids"][start:end]
        window_sizes.append(end - start)
        labels.append(torch.tensor(train_docs[d]["y"][start:end]))
    ids, lengths = pad_sentences(sentences, pad_id)
    return ids, lengths, window_sizes, pad_sequence(labels, batch_first=True, padding_value=-100)

def evaluate(model, docs, window=None):
    y_true, y_pred = [], []
    for doc in docs:
        with torch.autocast("cuda", dtype=amp_dtype):
            logits = model.predict(doc["ids"], pad_id, eval_batch_size, window)
        y_pred += logits.argmax(-1).tolist()
        y_true += doc["y"]
    return f1_score(y_true, y_pred, average="macro", zero_division=0), y_true, y_pred

steps_per_epoch = math.ceil(len(train_windows) / batch_windows / accum_steps)
print(f"{len(train_windows)} train_windows | {steps_per_epoch} optimizer steps per epoch")

weights = np.where(label_counts > 0, np.maximum(label_counts, 1) ** -class_weight_power, 1.0)
weights = weights * label_counts.sum() / (label_counts * weights).sum()
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)


def make_optimizer(model, total_steps):
    lorma_params = [p for m in model.modules() if isinstance(m, LoRMALinear) for p in (m.A, m.B)]
    lorma_ids = {id(p) for p in lorma_params}
    head_params = [p for p in model.parameters() if p.requires_grad and id(p) not in lorma_ids]

    optimizer = torch.optim.AdamW([
        {"params": lorma_params, "lr": lr_lorma, "weight_decay": 0.0},
        {"params": head_params, "lr": lr_head, "weight_decay": weight_decay},
    ])

    warmup_steps = int(warmup_frac * total_steps)
    def lr_factor(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        return (total_steps - step) / (total_steps - warmup_steps)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    return optimizer, scheduler, lorma_params + head_params


def distillation_loss(student_logits, student_vecs, teacher_logits, teacher_vecs, labels):
    keep = labels != -100
    s_logits = student_logits.float()[keep]
    t_logits = teacher_logits.float()[keep]
    y = labels[keep]

    hard = F.cross_entropy(s_logits, y, weight=class_weights)

    soft = F.kl_div(F.log_softmax(s_logits / kd_temperature, dim=-1),
                    F.softmax(t_logits / kd_temperature, dim=-1),
                    reduction="batchmean") * kd_temperature ** 2

    feature = (1 - F.cosine_similarity(student_vecs.float(), teacher_vecs.float(), dim=-1)).mean()

    total = kd_alpha * hard + (1 - kd_alpha) * soft + feature_weight * feature
    return total, hard, soft, feature


def train_model(model, epochs, teacher=None):
    total_steps = steps_per_epoch * epochs
    optimizer, scheduler, trainable = make_optimizer(model, total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

    best_f1, best_state, epochs_without_gain = -1.0, None, 0
    step, history, start_time = 0, [], time.time()

    for epoch in range(epochs):
        model.train()
        order = np.random.permutation(len(train_windows))

        for batch_num, i in enumerate(range(0, len(order), batch_windows), start=1):
            ids, lengths, window_sizes, labels = make_batch([train_windows[j] for j in order[i:i + batch_windows]])
            ids, lengths, labels = ids.to(device), lengths.to(device), labels.to(device)

            with torch.autocast("cuda", dtype=amp_dtype):
                logits, vectors = model(ids, lengths, window_sizes, return_vectors=True)
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits, teacher_vectors = teacher(ids, lengths, window_sizes, return_vectors=True)

            if teacher is None:
                loss = F.cross_entropy(logits.float().reshape(-1, num_labels), labels.reshape(-1),
                                       weight=class_weights, ignore_index=-100)
                history.append([loss.item()])
            else:
                loss, hard, soft, feature = distillation_loss(logits, vectors, teacher_logits, teacher_vectors, labels)
                history.append([loss.item(), hard.item(), soft.item(), feature.item()])

            scaler.scale(loss / accum_steps).backward()

            last_batch = i + batch_windows >= len(order)
            if batch_num % accum_steps == 0 or last_batch:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(trainable, max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
                step += 1

                if step % log_every == 0:
                    minutes = (time.time() - start_time) / 60
                    avg = np.mean(history[-log_every * accum_steps:], axis=0)
                    text = f"loss {avg[0]:.4f}"
                    if teacher is not None:
                        text += f" (hard {avg[1]:.3f} | soft {avg[2]:.3f} | feature {avg[3]:.3f})"
                    print(f"epoch {epoch + 1} | step {step}/{total_steps} | {text} | {minutes:.1f} min", flush=True)

        dev_f1, _, _ = evaluate(model, dev_docs)
        print(f"epoch {epoch + 1} dev macro-F1: {dev_f1:.4f}", flush=True)

        if dev_f1 > best_f1:
            best_f1, epochs_without_gain = dev_f1, 0
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
        else:
            epochs_without_gain += 1
            if epochs_without_gain == patience:
                break

    model.load_state_dict(best_state, strict=False)
    print(f"best dev macro-F1: {best_f1:.4f}")
    return best_f1

561 windows | 141 optimizer steps per epoch


In [11]:
if os.path.exists(teacher_save_path) and not retrain_teacher:
    print("loading the saved teacher")
    teacher.load_state_dict(torch.load(teacher_save_path), strict=False)
else:
    train_model(teacher, num_epochs)
    torch.save({n: p.detach().cpu() for n, p in teacher.named_parameters() if p.requires_grad}, teacher_save_path)

teacher.eval()
teacher.requires_grad_(False)

teacher_f1, _, _ = evaluate(teacher, dev_docs)
print(f"teacher dev macro-F1: {teacher_f1:.4f}")

epoch 1 | step 25/1410 | loss 2.3672 | 0.2 min
epoch 1 | step 50/1410 | loss 1.9671 | 0.5 min
epoch 1 | step 75/1410 | loss 1.7885 | 0.7 min
epoch 1 | step 100/1410 | loss 1.5781 | 0.9 min
epoch 1 | step 125/1410 | loss 1.5836 | 1.1 min
epoch 1 dev macro-F1: 0.4458
epoch 2 | step 150/1410 | loss 1.3962 | 1.5 min
epoch 2 | step 175/1410 | loss 1.1119 | 1.7 min
epoch 2 | step 200/1410 | loss 1.0826 | 2.0 min
epoch 2 | step 225/1410 | loss 1.1232 | 2.2 min
epoch 2 | step 250/1410 | loss 1.0913 | 2.5 min
epoch 2 | step 275/1410 | loss 0.9607 | 2.8 min
epoch 2 dev macro-F1: 0.4851
epoch 3 | step 300/1410 | loss 0.7615 | 3.1 min
epoch 3 | step 325/1410 | loss 0.7305 | 3.4 min
epoch 3 | step 350/1410 | loss 0.5372 | 3.6 min
epoch 3 | step 375/1410 | loss 0.6020 | 3.9 min
epoch 3 | step 400/1410 | loss 0.6527 | 4.1 min
epoch 3 dev macro-F1: 0.5547
epoch 4 | step 425/1410 | loss 0.6103 | 4.5 min
epoch 4 | step 450/1410 | loss 0.4187 | 4.7 min
epoch 4 | step 475/1410 | loss 0.3433 | 5.0 min
epoc

In [12]:
if use_kd:
    student_layers = np.linspace(0, num_teacher_layers - 1, num_student_layers).round().astype(int).tolist()
    student = build_classifier(student_layers)

    student_f1 = train_model(student, num_epochs, teacher=teacher)
    final_model, final_layers = student, student_layers
else:
    final_model, final_layers = teacher, list(range(num_teacher_layers))

12 decoder layers [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] | LoRMA on 24 linears | trainable 3,060,237 of 118,197,261 (2.59%)
epoch 1 | step 25/1410 | loss 4.7875 (hard 2.396 | soft 6.843 | feature 0.336) | 0.4 min
epoch 1 | step 50/1410 | loss 4.2475 (hard 2.244 | soft 5.934 | feature 0.317) | 0.8 min
epoch 1 | step 75/1410 | loss 3.8047 (hard 2.128 | soft 5.240 | feature 0.242) | 1.2 min
epoch 1 | step 100/1410 | loss 3.4569 (hard 1.890 | soft 4.873 | feature 0.151) | 1.6 min
epoch 1 | step 125/1410 | loss 2.9091 (hard 1.658 | soft 4.033 | feature 0.128) | 2.0 min
epoch 1 dev macro-F1: 0.4352
epoch 2 | step 150/1410 | loss 2.7055 (hard 1.571 | soft 3.731 | feature 0.109) | 2.4 min
epoch 2 | step 175/1410 | loss 2.2302 (hard 1.226 | soft 3.140 | feature 0.094) | 2.8 min
epoch 2 | step 200/1410 | loss 2.1038 (hard 1.239 | soft 2.883 | feature 0.086) | 3.2 min
epoch 2 | step 225/1410 | loss 2.0911 (hard 1.284 | soft 2.816 | feature 0.082) | 3.6 min
epoch 2 | step 250/1410 | loss 1.9825 (h

In [13]:
print(f"teacher dev macro-F1: {teacher_f1:.4f}  ({len(teacher.decoder.blocks)} layers)")
if use_kd:
    print(f"student dev macro-F1: {student_f1:.4f}  ({len(student.decoder.blocks)} layers)")
print("The student is used for the submission.\n" if use_kd else "")

f1_whole, y_true, y_pred = evaluate(final_model, dev_docs)
print(f"dev macro-F1: {f1_whole:.4f}")
f1_windowed = 0.0
if use_bigru:
    f1_windowed, _, _ = evaluate(final_model, dev_docs, window=window_size)
    print(f"BiGRU over train_windows of {window_size}: {f1_windowed:.4f} (whole document: {f1_whole:.4f})")
print()
print(classification_report(y_true, y_pred, labels=list(range(num_labels)), target_names=label_names, zero_division=0, digits=3))

teacher dev macro-F1: 0.5576  (12 layers)
student dev macro-F1: 0.5567  (12 layers)
The student is used for the submission.

dev macro-F1: 0.5567
BiGRU over windows of 32: 0.5495 (whole document: 0.5567)

                        precision    recall  f1-score   support

                  Fact      0.655     0.842     0.737       444
                 Issue      0.846     0.733     0.786        15
    ArgumentPetitioner      0.677     0.492     0.570       400
    ArgumentRespondent      0.584     0.522     0.551       272
               Statute      0.894     0.700     0.785       120
               Dissent      0.000     0.000     0.000         0
   PrecedentReliedUpon      0.493     0.654     0.562       254
PrecedentNotReliedUpon      0.412     0.280     0.333        50
    PrecedentOverruled      0.000     0.000     0.000         0
    RulingByLowerCourt      0.425     0.471     0.447       102
    RatioOfTheDecision      0.513     0.489     0.501       403
  RulingByPresentCourt    

In [14]:
test_window = None if f1_whole >= f1_windowed else window_size

predictions = {}
for doc in test_docs:
    with torch.autocast("cuda", dtype=amp_dtype):
        logits = final_model.predict(doc["ids"], pad_id, eval_batch_size, test_window)
    predictions[doc["id"]] = [label_names[i] for i in logits.argmax(-1).tolist()]

prediction_path = os.path.join(output_dir, "prediction.json")
zip_path = os.path.join(output_dir, "submission.zip")
with open(prediction_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(prediction_path, arcname="prediction.json")

print(f"{len(predictions)} documents -> {zip_path}")
for label, n in collections.Counter(l for p in predictions.values() for l in p).most_common():
    print(f"{label:<24s}{n:>8d}")

10 documents -> /kaggle/working/phase3/submission.zip
RatioOfTheDecision           644
PrecedentReliedUpon          498
Fact                         439
ArgumentPetitioner           255
Statute                      133
RulingByLowerCourt            97
ArgumentRespondent            52
RulingByPresentCourt          34
None                          24
Issue                         15
PrecedentNotReliedUpon         6


In [15]:
merge_lorma(final_model.decoder)
final_model.eval()

torch.save(final_model.decoder.state_dict(), os.path.join(output_dir, "phase3_decoder_merged.bin"))
head_state = {k: v.cpu() for k, v in final_model.state_dict().items() if not k.startswith("decoder.")}
torch.save(head_state, os.path.join(output_dir, "phase3_head.pt"))

with open(os.path.join(output_dir, "decoder_layers.json"), "w") as f:
    json.dump({"n_layers": len(final_layers), "kept_phase1_layers": final_layers}, f)